In [ ]:
#!/usr/bin/env python3
"""
Project 13 — Path A, Part 1: atlas-wide cis-decodability map + the law test.
Run on Kaggle (CPU is enough -- no GPU needed, no new model inference).
Companion: docs/Project13_v2_program.md, docs/TASK_pathA_kaggle.md, STATE.md (v2 banner).

WHAT THIS COMPUTES (headline deliverable of Path A)
-------------------------------------------------------
Per-cell-type cis-decodability across ALL 8,856 Decima pseudobulks
(201 cell types x 271 tissues x 82 diseases), using Decima's SHIPPED
test_pearson (held-out-gene Pearson correlation -- no new inference, it's
already in metadata.h5ad). Confound-controlled (depth, n_cells, study/batch,
expression breadth), measured against a reproducibility ceiling, then tested
for biological structure (the "law": does decodability track developmental
stage / lineage / specialization / proliferation?).

This is "Part 1" of Path A specifically because it is CHEAP: everything it
needs (test_pearson, n_cells, total_counts, study, cell_type, the 4
replicate prediction matrices for a reproducibility ceiling, and the
predicted expression profiles themselves for cell-type annotation) already
ships in the public `Genentech/decima-data/metadata.h5ad` (~3.3GB). No
external data pull, no GPU. It always fits in a Kaggle session (~10-20 min).
Part 2 (heavy, CZ CELLxGene census pulls for measured-data validation) is a
separate notebook -- see kaggle_pathA_part2_identity_gap_validation.py and
docs/pathA_part2_shard_plan.md.

Pipeline (mirrors the validated sandbox prototype; see STATE.md RESULTS LOG
"Path A Part 1" for the reference run's numbers)
-------------------------------------------------------------------------------
1. Load obs metadata for all 8,856 pseudobulks (cell_type, tissue, disease,
   organ, study, n_cells, n_genes, total_counts, test_pearson).
2. CONFOUND CONTROL: OLS  test_pearson ~ log10(n_cells) + log10(total_counts)
   + n_genes + C(study)  (study absorbed as fixed effects -- 241 studies,
   241-1 dummies, well-determined with n=8,856). Residual =
   decodability_controlled, the confound-controlled decodability per
   pseudobulk. Also fit a depth-only model (no study FE) for comparison.
3. REPRODUCIBILITY CEILING (two independent checks, so "not decodable" is
   not conflated with "the model/data is just noisy here"):
   (a) Model-internal: mean pairwise Pearson correlation across Decima's 4
       replicate prediction matrices (v1_rep0..3) per pseudobulk -- a
       model-self-consistency ceiling, needs zero new inference.
   (b) Cross-instance split-half: for cell types with >=4 pseudobulk
       instances (different tissue/disease/study), randomly split instances
       into two halves, average decodability_controlled per half, Spearman-
       correlate across cell types. Tests whether the PER-CELL-TYPE RANKING
       is reproducible, not just whether any single number is noisy.
4. CELL-TYPE ANNOTATION (documented approximations, not a curated ontology
   -- flagged as a limitation, not hidden):
   - specialization_score: Yanai's Tau tissue-specificity index computed
     per gene across the 201 cell-type-mean predicted-expression profiles
     (aggregated from the `preds` layer, again already-shipped, no new
     inference), summed over each cell type's Tau-weighted "own" marker
     genes (genes for which that cell type is the arg-max AND Tau>0.8).
   - proliferation_score_z: mean predicted log1p expression of 8 canonical
     proliferation markers (MKI67, TOP2A, PCNA, CCNB1, CDK1, BUB1, CCNB2,
     AURKB; hardcoded stable Ensembl IDs), z-scored across cell types.
   - lineage: keyword/regex classification of the cell_type name into
     {immune, neural, epithelial, endothelial, stromal, muscle,
     germline_reproductive, pigment, stem_progenitor, other/unclassified}.
     This is NOT a formal Cell Ontology mapping -- a full CL-based
     annotation of 201 free-text labels spanning the whole body was judged
     out of scope; the regex classifier is transparent, inspectable, and
     was manually patched until 0/201 cell types were left unclassified in
     the reference run (see the PATCHES dict below for the exact overrides
     applied) -- but it can misclassify ambiguous names.
   - stem_progenitor_flag: boolean, name contains stem/progenitor/
     precursor/immature/naive/proliferating/cycling/HSC/blast.
5. THE LAW TEST: OLS  decodability_controlled_mean ~ specialization_score +
   proliferation_score_z + stem_progenitor_flag + C(lineage), plus simple
   Spearman correlations and a stem-vs-rest Mann-Whitney, plus per-lineage
   means. Go/no-go: (a) reproducible (yes/no) AND (b) biologically
   structured (yes/no/partial) -- see the go_no_go block in the output JSON.

CRITICAL CAVEAT (do not drop this from any write-up)
-------------------------------------------------------
In the reference run, immune cell types scored HIGHEST on confound-controlled
decodability and epithelial cell types scored LOWEST. Immune/blood is
exactly the lineage this project's Chapter 1 (and plausibly Decima's own
training/curation emphasis, given its eQTL-validation use case) studied most
intensively. This law test CANNOT distinguish "immune biology is genuinely
more cis-decodable" from "Decima has simply seen more/better-characterized
immune regulatory elements." Flagged, not resolved.

Output
------
    data/processed/atlas_obs.parquet                         all 8,856 pseudobulks' metadata
    data/processed/atlas_decodability_pseudobulk.parquet      + decodability_controlled/raw/depth_only, replicate_agreement
    data/processed/atlas_celltype_annotations.parquet          201 cell types x (specialization/proliferation/lineage/stem flag)
    data/processed/atlas_celltype_decodability_map.parquet     the headline map: per-cell-type decodability + annotations
    data/processed/atlas_law_test_report.json                  full stats + go/no-go verdict

Usage
-----
    On Kaggle: New Notebook -> enable Internet -> paste this file into a
    cell (or upload + `!python this_file.py`) -> Run All. Downloads
    metadata.h5ad from HuggingFace on first run if not already present
    locally (~3.3GB, few minutes). CPU only, ~10,000 obs is small.
"""
from __future__ import annotations
import json, os, re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

# ---------------------------------------------------------------
# Paths -- on Kaggle, adjust OUT_DIR to /kaggle/working; metadata.h5ad is
# downloaded to /kaggle/working/data/raw if not already present.
# ---------------------------------------------------------------
IS_KAGGLE = Path("/kaggle").exists()
REPO = Path("/kaggle/working") if IS_KAGGLE else Path(__file__).resolve().parent.parent
RAW = REPO / "data" / "raw"
PROC = REPO / "data" / "processed"
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)
METADATA_H5AD = RAW / "metadata.h5ad"

PROLIF_GENES = {
    "MKI67": "ENSG00000148773", "TOP2A": "ENSG00000131747", "PCNA": "ENSG00000132646",
    "CCNB1": "ENSG00000134057", "CDK1": "ENSG00000170312", "BUB1": "ENSG00000169679",
    "CCNB2": "ENSG00000157456", "AURKB": "ENSG00000178999",
}

LINEAGE_PATTERNS = [
    ("immune", r"(T cell|B cell|\bNK\b|natural killer|macrophage|\bM(Φ|Phi)\b|monocyte|dendritic|\bDC\b|mast cell|neutrophil|plasma\b|plasmablast|lymphocyte|myeloid|granulocyte|eosinophil|basophil|\bTreg\b|\bTfh\b|\bILC\b|innate lymphoid|leukocyte|immune|\bCTL\b|CD[48]\+|Langerhans|erythroid|megakaryocyte|platelet|^T [A-Z]|cycling T\b|Mast\b)"),
    ("neural", r"(neuron|glia\b|astrocyte|oligodendrocyte|microglia|Schwann|neural|Purkinje|granule cell|interneuron|excitatory|inhibitory|amacrine|bipolar|horizontal cell|ganglion cell|Mueller|ependymal|rhombic lip|hippocampal|cortic|intratelencephalic|Chandelier|choroid plexus|neuroendocrine|rod cell|cone cell|retinal)"),
    ("epithelial", r"(epithelial|keratinocyte|\bKC\b|enterocyte|hepatocyte|alveolar|ductal|acinar|goblet|Paneth|urothelial|epithelium|pneumocyte|cholangiocyte|basal cell|ciliated|club cell|ionocyte|secretory cell|hair follicle|sebaceous|sweat gland|cornified|respiratory basal|mucus|enteroendocrine)"),
    ("endothelial", r"(endothelial|\bEC\b|vascular endothel|\bLEC\b|\bVEC\b|lymphatic endothel|^Vascular$)"),
    ("stromal", r"(fibroblast|stromal|mesenchymal|pericyte|smooth muscle|\bSMC\b|myofibroblast|mesothelial)"),
    ("muscle", r"(myocyte|cardiomyocyte|muscle fiber|skeletal muscle|myofiber)"),
    ("germline_reproductive", r"(oocyte|spermat|granulosa|theca cell|Leydig|Sertoli)"),
    ("pigment", r"(melano|Melano)"),
    ("stem_progenitor", r"(stem cell|progenitor|precursor|\bHSC\b|blast\b)"),
]
# manual patches for the handful of names the regex still missed in the reference run
LINEAGE_PATCHES = {
    "Deep-layer near-projecting": "neural", "Mammillary body": "neural",
    "OFFx cell": "neural", "Splatter": "neural",
}
STEM_FLAG_PATTERN = r"\b(stem|progenitor|precursor|immature|naive|proliferating|cycling|HSC|blast)\b"


def log(m):
    print(f"[pathA-part1] {m}", flush=True)


def ensure_metadata():
    if METADATA_H5AD.exists():
        log(f"metadata.h5ad already present at {METADATA_H5AD}")
        return
    log("downloading metadata.h5ad from HuggingFace (Genentech/decima-data, ~3.3GB)...")
    from huggingface_hub import hf_hub_download
    path = hf_hub_download(repo_id="Genentech/decima-data", filename="metadata.h5ad",
                            repo_type="dataset", local_dir=str(RAW))
    if Path(path) != METADATA_H5AD:
        Path(path).rename(METADATA_H5AD)
    log("download complete")


def decode_bytes(arr):
    return np.array([x.decode("utf-8", errors="replace") if isinstance(x, bytes) else x for x in arr])


def cat_col(obs, name):
    g = obs[name]
    if hasattr(g, "keys"):
        cats = decode_bytes(g["categories"][:])
        return cats[g["codes"][:]]
    return g[:]


def load_obs():
    import h5py
    f = h5py.File(METADATA_H5AD, "r")
    obs = f["obs"]
    df = pd.DataFrame({
        "cell_type": cat_col(obs, "cell_type"), "celltype_coarse": cat_col(obs, "celltype_coarse"),
        "dataset": cat_col(obs, "dataset"), "disease": cat_col(obs, "disease"),
        "organ": cat_col(obs, "organ"), "region": cat_col(obs, "region"),
        "study": cat_col(obs, "study"), "subregion": cat_col(obs, "subregion"),
        "tissue": cat_col(obs, "tissue"),
        "n_cells": obs["n_cells"][:], "n_genes": obs["n_genes"][:],
        "size_factor": obs["size_factor"][:], "test_pearson": obs["test_pearson"][:],
        "total_counts": obs["total_counts"][:], "train_pearson": obs["train_pearson"][:],
        "val_pearson": obs["val_pearson"][:],
    })
    f.close()
    return df


def confound_control(df):
    log("confound control: OLS test_pearson ~ depth + n_genes + study fixed effects")
    X = pd.DataFrame({
        "log_n_cells": np.log10(df["n_cells"].clip(lower=1)),
        "log_total_counts": np.log10(df["total_counts"].clip(lower=1)),
        "n_genes": df["n_genes"],
    })
    study_dummies = pd.get_dummies(df["study"], prefix="study", drop_first=True, dtype=float)
    Xfull = pd.concat([X, study_dummies], axis=1)
    Xfull.insert(0, "const", 1.0)
    y = df["test_pearson"].values
    Xm = Xfull.values.astype(np.float64)
    beta, _, rank, _ = np.linalg.lstsq(Xm, y, rcond=None)
    resid = y - Xm @ beta
    log(f"  full model: rank={rank}/{Xm.shape[1]}, R2={1 - resid.var()/y.var():.4f}")
    df["decodability_raw"] = y
    df["decodability_controlled"] = resid

    X2 = Xfull[[c for c in Xfull.columns if not c.startswith("study_")]].values.astype(np.float64)
    beta2, *_ = np.linalg.lstsq(X2, y, rcond=None)
    resid2 = y - X2 @ beta2
    df["decodability_depth_only"] = resid2
    log(f"  depth-only model: R2={1 - resid2.var()/y.var():.4f}")
    return df


def replicate_agreement(df):
    log("reproducibility ceiling (a): inter-replicate agreement (v1_rep0..3), chunked, no new inference")
    import h5py
    f = h5py.File(METADATA_H5AD, "r")
    reps = [f["layers"][f"v1_rep{i}"] for i in range(4)]
    n = len(df)
    chunk = 500
    agree = np.zeros(n, dtype=np.float32)
    for start in range(0, n, chunk):
        end = min(start + chunk, n)
        mats = np.stack([r[start:end, :] for r in reps], axis=0)  # (4, chunk, genes)
        mats_c = mats - mats.mean(axis=2, keepdims=True)
        norms = np.sqrt((mats_c ** 2).sum(axis=2))
        pair_corrs = []
        for i in range(4):
            for j in range(i + 1, 4):
                num = (mats_c[i] * mats_c[j]).sum(axis=1)
                den = norms[i] * norms[j]
                den[den == 0] = np.nan
                pair_corrs.append(num / den)
        agree[start:end] = np.nanmean(np.stack(pair_corrs, axis=0), axis=0)
    f.close()
    df["replicate_agreement"] = agree
    log(f"  mean={np.nanmean(agree):.4f}, median={np.nanmedian(agree):.4f}")
    return df


def annotate_cell_types(df):
    import h5py
    f = h5py.File(METADATA_H5AD, "r")
    preds = f["layers"]["preds"]
    gene_id = f["var"]["gene_id"][:].astype(str)
    n_genes = len(gene_id)

    cell_types = sorted(df["cell_type"].unique())
    ct_index = {ct: i for i, ct in enumerate(cell_types)}
    ct_code = df["cell_type"].map(ct_index).values

    log(f"aggregating preds to {len(cell_types)} cell-type-mean profiles (streamed)")
    sums = np.zeros((len(cell_types), n_genes), dtype=np.float64)
    counts = np.zeros(len(cell_types), dtype=np.int64)
    chunk = 800
    for start in range(0, preds.shape[0], chunk):
        end = min(start + chunk, preds.shape[0])
        block = preds[start:end, :].astype(np.float64)
        for i, c in enumerate(ct_code[start:end]):
            sums[c] += block[i]
            counts[c] += 1
    ct_profile = sums / counts[:, None]
    f.close()

    log("Tau specificity + specialization score")
    lin = np.expm1(np.clip(ct_profile, 0, None))
    maxv = lin.max(axis=0)
    maxv_safe = np.where(maxv > 0, maxv, 1.0)
    tau = (1 - lin / maxv_safe[None, :]).sum(axis=0) / (len(cell_types) - 1)
    tau[maxv == 0] = np.nan
    argmax_ct = lin.argmax(axis=0)
    specialization_score = np.zeros(len(cell_types))
    marker_count = np.zeros(len(cell_types), dtype=int)
    for g in np.where(~np.isnan(tau))[0]:
        c = argmax_ct[g]
        specialization_score[c] += tau[g]
        if tau[g] > 0.8:
            marker_count[c] += 1

    log("proliferation score")
    gidx = {g: i for i, g in enumerate(gene_id)}
    prolif_idx = [gidx[e] for e in PROLIF_GENES.values() if e in gidx]
    log(f"  markers found: {len(prolif_idx)}/{len(PROLIF_GENES)}")
    prolif_raw = ct_profile[:, prolif_idx].mean(axis=1)
    proliferation_score = (prolif_raw - prolif_raw.mean()) / prolif_raw.std()

    def classify_lineage(name):
        if name in LINEAGE_PATCHES:
            return LINEAGE_PATCHES[name]
        for lin_name, pat in LINEAGE_PATTERNS:
            if re.search(pat, name, flags=re.IGNORECASE):
                return lin_name
        return "other/unclassified"

    lineage = [classify_lineage(ct) for ct in cell_types]
    stem_flag = [bool(re.search(STEM_FLAG_PATTERN, ct, re.IGNORECASE)) for ct in cell_types]

    ann = pd.DataFrame({
        "cell_type": cell_types, "specialization_score": specialization_score,
        "marker_count_tau0.8": marker_count, "proliferation_score_z": proliferation_score,
        "lineage": lineage, "stem_progenitor_flag": stem_flag, "n_pseudobulk_instances": counts,
    })
    n_unclassified = (ann["lineage"] == "other/unclassified").sum()
    log(f"lineage classification: {n_unclassified}/{len(ann)} unclassified "
        f"({ann[ann.lineage=='other/unclassified']['cell_type'].tolist() if n_unclassified else 'none'})")
    return ann


def law_test(pb, ann):
    import statsmodels.formula.api as smf

    agg = pb.groupby("cell_type").agg(
        decodability_controlled_mean=("decodability_controlled", "mean"),
        decodability_raw_mean=("decodability_raw", "mean"),
        replicate_agreement_mean=("replicate_agreement", "mean"),
        n_instances=("decodability_controlled", "size"),
    ).reset_index()

    log("cross-instance split-half reproducibility (cell types with >=4 instances)")
    rng = np.random.RandomState(42)
    half_a, half_b, cts_used = [], [], []
    for ct, sub in pb.groupby("cell_type"):
        if len(sub) < 4:
            continue
        idx = rng.permutation(len(sub))
        half_a.append(sub.iloc[idx[:len(idx)//2]]["decodability_controlled"].mean())
        half_b.append(sub.iloc[idx[len(idx)//2:]]["decodability_controlled"].mean())
        cts_used.append(ct)
    rho_split, p_split = stats.spearmanr(half_a, half_b)
    log(f"  n={len(cts_used)} cell types, Spearman rho={rho_split:.3f}, p={p_split:.3g}")

    merged = agg.merge(ann, on="cell_type", how="left")

    log("law test: OLS decodability ~ specialization + proliferation + lineage + stem flag")
    m = merged.dropna(subset=["decodability_controlled_mean", "specialization_score", "proliferation_score_z"]).copy()
    m["stem_progenitor_flag"] = m["stem_progenitor_flag"].astype(int)
    model = smf.ols("decodability_controlled_mean ~ specialization_score + proliferation_score_z + "
                     "stem_progenitor_flag + C(lineage)", data=m).fit()
    log(model.summary().as_text())

    r_spec, p_spec = stats.spearmanr(m["decodability_controlled_mean"], m["specialization_score"])
    r_prolif, p_prolif = stats.spearmanr(m["decodability_controlled_mean"], m["proliferation_score_z"])
    stem_test = stats.mannwhitneyu(
        m[m.stem_progenitor_flag == 1]["decodability_controlled_mean"],
        m[m.stem_progenitor_flag == 0]["decodability_controlled_mean"], alternative="two-sided")
    lineage_means = m.groupby("lineage")["decodability_controlled_mean"].agg(
        ["mean", "median", "std", "count"]).sort_values("mean")

    report = {
        "n_cell_types": len(merged), "n_pseudobulks": len(pb),
        "confound_control": {"ols_full_R2": float(1 - pb["decodability_controlled"].var() / pb["decodability_raw"].var())},
        "reproducibility_ceiling": {
            "replicate_agreement_mean": float(pb["replicate_agreement"].mean()),
            "replicate_agreement_median": float(pb["replicate_agreement"].median()),
            "cross_instance_splithalf_n_celltypes": len(cts_used),
            "cross_instance_splithalf_spearman_rho": float(rho_split),
            "cross_instance_splithalf_p": float(p_split),
        },
        "law_test": {
            "ols_r2": float(model.rsquared), "ols_r2_adj": float(model.rsquared_adj),
            "specialization_p": float(model.pvalues.get("specialization_score", np.nan)),
            "proliferation_p": float(model.pvalues.get("proliferation_score_z", np.nan)),
            "stem_progenitor_p": float(model.pvalues.get("stem_progenitor_flag", np.nan)),
            "spearman_decodability_vs_specialization": {"rho": float(r_spec), "p": float(p_spec)},
            "spearman_decodability_vs_proliferation": {"rho": float(r_prolif), "p": float(p_prolif)},
            "stem_vs_rest_mannwhitney_p": float(stem_test.pvalue),
            "lineage_means": lineage_means.to_dict(orient="index"),
        },
    }
    return merged, report


def main():
    ensure_metadata()
    df = load_obs()
    df.to_parquet(PROC / "atlas_obs.parquet")
    df = confound_control(df)
    df = replicate_agreement(df)
    df.to_parquet(PROC / "atlas_decodability_pseudobulk.parquet")

    ann = annotate_cell_types(df)
    ann.to_parquet(PROC / "atlas_celltype_annotations.parquet")

    merged, report = law_test(df, ann)
    merged.to_parquet(PROC / "atlas_celltype_decodability_map.parquet")
    (PROC / "atlas_law_test_report.json").write_text(json.dumps(report, indent=2, default=str))
    log(f"DONE. Wrote atlas_celltype_decodability_map.parquet ({len(merged)} cell types) "
        f"and atlas_law_test_report.json")


if __name__ == "__main__":
    main()